# Agent — 智能体（create_agent）

LangChain 1.0 新增 `create_agent`，替代旧的 AgentExecutor，是构建智能体的新标准入口。

In [1]:
from langchain_learning.llm import get_llm
from langchain.agents import create_agent

llm = get_llm()

**术语：**
- **Agent（智能体）** = 能自动决定调什么工具、怎么调、什么时候给最终回答的程序
- **create_agent** = LangChain 1.0 创建智能体的入口函数
- **system_prompt** = 设定 Agent 的角色和行为
- **content_blocks** = LangChain 1.0 统一的消息内容格式，支持文本、工具调用等

### 1. 定义工具

In [2]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气"""
    return f"{city} 今天晴天，气温 22-28°C"

@tool
def calculate(expr: str) -> float:
    """计算数学表达式"""
    try:
        return eval(expr)
    except:
        return 0.0

print("工具定义完成")
print(get_weather.invoke({"city": "北京"}))

工具定义完成
北京 今天晴天，气温 22-28°C


**术语：**
- **@tool** = 装饰器，把函数变成工具
- 工具需要有清晰的名称、docstring 描述、类型注解——这些都会被 LLM 用来决定何时调用

### 2. 使用 create_agent 创建智能体

In [3]:
agent = create_agent(
    model=llm,
    tools=[get_weather, calculate],
    system_prompt="你是一个智能助手，可以使用工具查询天气和计算数学。",
)

result = agent.invoke({"messages": [{"role": "user", "content": "北京天气怎么样？"}]})
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content[:80] if hasattr(msg, 'content') and msg.content else msg.content_blocks}")

human: 北京天气怎么样？
ai: 好的，我来查询一下北京的天气情况。
tool: 北京 今天晴天，气温 22-28°C
ai: 北京今天天气**晴朗**，气温在 **22°C 到 28°C** 之间，天气不错，适合外出活动哦！☀️


**术语：**
- **create_agent(model, tools, system_prompt)** = 三步创建一个智能体
- **invoke({"messages": [...]})** = 传入对话消息
- **result["messages"]** = 包含完整的对话历史，包括工具的调用和结果
- Agent 会自动：理解问题 → 决定调用工具 → 获取结果 → 生成最终回答

### 3. 多次推理——Agent 自动决定调用顺序

In [6]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "(85 + 17) * 2 等于多少？再帮我查一下上海的天气。"}]
})

print("=== 完整对话 ===")
for msg in result["messages"]:
    if msg.type == "ai":
        for block in msg.content_blocks:
            if block["type"] == "text":
                print(f"[AI]: {block['text'][:100]}")
            elif block["type"] == "tool_use":
                print(f"[调用工具] {block['name']}({block['input']})")
    elif msg.type == "tool_result":
        print(f"[工具结果] {msg.content[:80]}")

=== 完整对话 ===
[AI]: 好的，我来同时处理这两个请求！
[AI]: 好的，结果如下：

1. **数学计算**：(85 + 17) × 2 = **204**
2. **上海天气**：今天 ☀️ 晴天，气温 **22~28°C**，天气不错哦！


**术语：**
- **content_blocks** = LangChain 1.0 的消息内容块，每条消息可以包含多个块
- **块类型**：`text`（文本）、`tool_use`（工具调用）、`tool_result`（工具结果）
- Agent 自动处理多步推理：先算数学题，再查天气，最后汇总回答

### 4. 流式输出 Agent

In [8]:
print("流式输出：")
for chunk in agent.stream({"messages": [{"role": "user", "content": "用一句话介绍广州的天气"}]}):
    # stream 输出 dict，格式如 {"model": {"messages": [...]}} 或 {"tools": {"messages": [...]}}
    if "model" in chunk:
        msg = chunk["model"]["messages"][0]
        if msg.content:
            print(msg.content, end="", flush=True)
print()


流式输出：
好的，我来查询广州的天气。广州今天天气晴朗，气温在22°C到28°C之间，温暖舒适，非常适合外出活动！


**术语：**
- **agent.stream()** = Agent 也支持流式输出，边运行边返回结果
- Agent 的 stream 会输出所有事件（思考、调工具、生成回答）

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant Tools
    
    User->>Agent: 提问
    Agent->>Agent: 理解问题
    Agent->>Tools: 调用工具
    Tools-->>Agent: 返回结果
    Agent->>Agent: 分析结果
    Agent-->>User: 最终回答
```

**术语：**
- Agent 的工作流程是循环的：思考 → 行动 → 观察 → 再思考 → ... → 最终回答
- 这个循环在 `create_agent` 内部自动处理，不需要手动编写循环逻辑